# Netflix Project — Real Data Analysis with Pandas

**Uses only:** Python (functions) · NumPy · Pandas
**Data:** `netflix_titles.csv` from Kaggle — https://www.kaggle.com/datasets/shivamb/netflix-shows (8,807 real titles)

---

### The Story
We joined **Netflix's content team** as a data analyst. Our manager hands you the full catalogue.

## What We'll Explore Today

We'll holding **Netflix's entire catalogue** — 8,807 movies and TV shows in one file. Think about it: what questions can a single CSV answer?

- Does Netflix have **more movies or more TV shows**?
- Some Filtering Problem:
  - Are there any **Bangladeshi films** on Netflix? Which ones?
  - How many titles were released after 2020?
  - What are the **5 newest releases**?
- Which **country** produces the most content? And where does our neighbour **India** rank?
- Which **genre** dominates — Drama, Comedy, or something else?
- In which **year** did Netflix add the most content?
- How long is a **typical movie**? What's the **longest movie**? Which show has the **most seasons**?
- Is Netflix's content mostly for **adults** or for **kids**?

### Import Libraries and the CSV

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("netflix_titles.csv")
print("Loaded:", df.shape[0], "titles,", df.shape[1], "columns")

Loaded: 8807 titles, 12 columns


### Mission 1 — Meet the Data

In [2]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [4]:
df.describe()

,release_year
count,8807.000000
mean,2014.180198
std,8.819312
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


In [5]:
# where are the gaps?
df.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

**Talk about it:** Real data is messy. `director` is missing for 2,634 titles, `country` for 831. A few rows are missing `date_added`, `rating`, `duration`. We must clean before we analyse — **garbage in, garbage out**.

### Mission 2 — Clean the Data

In [6]:
def fill_unknown(data, column):
    """Fill missing text values in one column with 'Unknown'."""
    data[column] = data[column].fillna("Unknown")
    return data

for col in ["director", "cast", "country"]:
    df = fill_unknown(df, col)

# only 17 rows are missing these -> drop them
df = df.dropna(subset=["date_added", "rating", "duration"])

print("Shape now:", df.shape)
print("Missing left:", df.isnull().sum().sum())

Shape now: (8790, 12)
Missing left: 0


**Challenge:** Look at `duration`. Movies say `"90 min"`, shows say `"2 Seasons"` — a number and a word mixed in one text column. We cannot do math on text. Therefore, Split it.

In [7]:
def split_duration(data):
    parts = data["duration"].str.split(" ")
    data["duration_num"] = parts.str[0].astype(int)
    data["duration_unit"] = parts.str[1]
    return data

df = split_duration(df)
df[["title", "type", "duration", "duration_num", "duration_unit"]].head()

,title,type,duration,duration_num,duration_unit
0,Dick Johnson Is Dead,Movie,90 min,90,min
1,Blood & Water,TV Show,2 Seasons,2,Seasons
2,Ganglands,TV Show,1 Season,1,Season
3,Jailbirds New Orleans,TV Show,1 Season,1,Season
4,Kota Factory,TV Show,2 Seasons,2,Seasons


**Challenge:** `date_added` is text like `"September 25, 2021"`. Turn it into a real **DATE** and pull out the **YEAR**.

In [8]:
df["date_added"] = pd.to_datetime(df["date_added"].str.strip())
df["year_added"] = df["date_added"].dt.year
df[["title", "date_added", "year_added"]].head()

,title,date_added,year_added
0,Dick Johnson Is Dead,2021-09-25,2021
1,Blood & Water,2021-09-24,2021
2,Ganglands,2021-09-24,2021
3,Jailbirds New Orleans,2021-09-24,2021
4,Kota Factory,2021-09-24,2021


### Mission 3 — Our QnA

##### 1. Does Netflix have **more movies or more TV shows**?

In [9]:
df["type"].value_counts()

type
Movie      6126
TV Show    2664
Name: count, dtype: int64

##### 2. Some Filtering Problem:
  - Are there any **Bangladeshi films** on Netflix? Which ones?
  - How many titles were released after 2020?
  - What are the **5 newest releases**?

In [10]:
# Are there any Bangladeshi films on Netflix? Which ones?
bd = df[df["country"].str.contains("Bangladesh")]
bd[["title", "type", "release_year"]]

,title,type,release_year
1337,Doob: No Bed of Roses,Movie,2017
3124,"Sincerely Yours, Dhaka",Movie,2018
4210,Komola Rocket,Movie,2018
6472,Chittagong,Movie,2012


In [11]:
# How many titles released after 2020?
recent = df[df["release_year"] > 2020]
print(len(recent), "titles released after 2020")

592 titles released after 2020


In [12]:
# The 5 newest releases
df.sort_values("release_year", ascending=False)[["title", "type", "release_year"]].head()

,title,type,release_year
1,Blood & Water,TV Show,2021
8437,The Netflix Afterparty,TV Show,2021
31,Chicago Party Aunt,TV Show,2021
30,Ankahi Kahaniya,Movie,2021
25,Love on the Spectrum,TV Show,2021


##### 3. Which **country** produces the most content? And where does our neighbour **India** rank?

In [13]:
df["main_country"] = df["country"].str.split(",").str[0].str.strip()

# ignore titles where the country is unknown
known = df[df["main_country"] != "Unknown"]
known["main_country"].value_counts().head(5)

main_country
United States     3202
India             1008
United Kingdom     627
Canada             271
Japan              257
Name: count, dtype: int64

##### 4. Which **genre** dominates — Drama, Comedy, or something else?

In [14]:
genre_count = {}

for genres in df["listed_in"]:
    for g in genres.split(","):
        g = g.strip()
        if g in genre_count:
            genre_count[g] = genre_count[g] + 1
        else:
            genre_count[g] = 1

genre_series = pd.Series(genre_count).sort_values(ascending=False)
genre_series.head(5)

International Movies      2752
Dramas                    2426
Comedies                  1674
International TV Shows    1349
Documentaries              869
dtype: int64

##### 5. In which **year** did Netflix add the most content?

In [15]:
df["year_added"].value_counts().head(5)

year_added
2019    2016
2020    1879
2018    1648
2021    1498
2017    1185
Name: count, dtype: int64

##### 6. How long is a **typical movie**? What's the **longest movie**? Which show has the **most seasons**?

In [16]:
movies = df[df["type"] == "Movie"]

avg_len = np.round(np.mean(movies["duration_num"]), 1)
print(f"Average movie length: {avg_len} minutes")

shortest_len = np.min(movies["duration_num"])
longest_len = np.max(movies["duration_num"])
print(f"Shortest: {shortest_len}, min |  Longest: {longest_len} min")

Average movie length: 99.6 minutes
Shortest: 3, min |  Longest: 312 min


In [17]:
# the longest movie
movies.sort_values("duration_num", ascending=False)[["title", "duration_num", "release_year"]].head(3)

,title,duration_num,release_year
4253,Black Mirror: Bandersnatch,312,2018
717,Headspace: Unwind Your Mind,273,2021
2491,The School of Mischief,253,1973


In [18]:
# the longest-running TV show (most seasons)
shows = df[df["type"] == "TV Show"]
shows.sort_values("duration_num", ascending=False)[["title", "duration_num"]].head(3)

,title,duration_num
548,Grey's Anatomy,17
2423,Supernatural,15
4798,NCIS,15


##### 7. Is Netflix's content mostly for **adults** or for **kids**?

In [19]:
df["rating"].value_counts().head(5)

rating
TV-MA    3205
TV-14    2157
TV-PG     861
R         799
PG-13     490
Name: count, dtype: int64

**Verdict:** TV-MA (mature) is the biggest rating — Netflix's catalogue is mostly for adults, not kids.

### The Report — What We Found

- Netflix is **~70% movies** (6,126 movies vs 2,664 shows).
- The **United States** leads, and **India is #2** in the world (1,008 titles).
- Top genres: **International Movies, Dramas, Comedies**.
- Netflix added the most titles in **2019** (2,016), then 2020.
- A typical movie is about **100 minutes**; the longest is *Black Mirror: Bandersnatch* at 312 min. *Grey's Anatomy* has the most seasons (17).
- Most content is rated **TV-MA** — made for adults.
- **4 Bangladeshi films** are on the platform.
